In [1]:
import os
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.pipeline import make_pipeline
from imblearn.pipeline import make_pipeline as imbl_pipe
from imblearn.over_sampling import SMOTE

In [2]:
abt = pd.read_csv("../data/churn_prepared_data.csv")

# Object for target variable
y = abt.Exited

# Object for input features
X = abt.drop(['Exited'], axis=1)

print(X.shape, y.shape)

(10000, 10) (10000,)


In [3]:
# List numerical features
num_columns = X.select_dtypes(include='number').columns.tolist()
num_columns

['CreditScore',
 'Age',
 'Tenure',
 'Balance',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember',
 'EstimatedSalary']

In [4]:
# List categorical features
cat_columns = X.select_dtypes(include='object').columns.tolist()
cat_columns

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_2612\210812281.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = X.select_dtypes(include='object').columns.tolist()


['Geography', 'Gender']

In [5]:
random_state = 10
# Split X and y into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.25,
                                                    random_state=random_state,
                                                    stratify=abt.Exited)

# Print number of observations in X_train, X_test, y_train, and y_test
print(len(X_train), len(X_test), len(y_train), len(y_test))

7500 2500 7500 2500


In [6]:
X_train = X_train.values
X_test = X_test.values

In [7]:
import joblib
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score

#Load models
rf_model = joblib.load('../models/rf_weighted.sav')
svm_model = joblib.load('../models/svm_weighted.sav')
lr_model = joblib.load('../models/lr_weighted.sav')
xgb_model = joblib.load('../models/xgb_weighted.sav')
dt_model = joblib.load('../models/dt_weighted.sav')

# Predict probabilities for each model
proba_rf = rf_model.predict_proba(X_test)[:, 1]
proba_svm = svm_model.predict_proba(X_test)[:, 1]
proba_lr = lr_model.predict_proba(X_test)[:, 1]
proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
proba_dt = dt_model.predict_proba(X_test)[:, 1]

# Soft Voting
ensemble_proba = (proba_rf + proba_svm + proba_xgb + proba_dt) / 4

# Calculate ROC-AUC score
auc_ensemble = roc_auc_score(y_test, ensemble_proba)

# Calculate F1-score by thresholding probabilities (e.g., if probability >= 0.5, predict as 1)
ensemble_pred = np.where(ensemble_proba >= 0.5, 1, 0)
f1_ensemble = f1_score(y_test, ensemble_pred)

# Print evaluation report
print("KẾT QUẢ ENSEMBLE MODEL (SOFT VOTING): ")
print(f"ROC-AUC : {auc_ensemble:.4f}")
print(f"F1-Score: {f1_ensemble:.4f}")

d:\DS102\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
d:\DS102\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


KẾT QUẢ ENSEMBLE MODEL (SOFT VOTING): 
ROC-AUC : 0.8704
F1-Score: 0.6346


In [8]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import warnings

# Generate the Classification Report and Confusion Matrix for the Ensemble Model
print("ENSEMBLE MODEL CLASSIFICATION REPORT: ")
report = classification_report(y_test, ensemble_pred, target_names=['Stay (0)', 'Churn (1)'])
print(report)

print("ENSEMBLE MODEL CONFUSION MATRIX: ")
cm = confusion_matrix(y_test, ensemble_pred)
cm = np.around(cm / cm.sum(axis=1)[:, np.newaxis], 2)
print(cm)

ENSEMBLE MODEL CLASSIFICATION REPORT: 
              precision    recall  f1-score   support

    Stay (0)       0.91      0.88      0.90      1991
   Churn (1)       0.60      0.67      0.63       509

    accuracy                           0.84      2500
   macro avg       0.76      0.78      0.77      2500
weighted avg       0.85      0.84      0.85      2500

ENSEMBLE MODEL CONFUSION MATRIX: 
[[0.88 0.12]
 [0.33 0.67]]


In [9]:
# ==========================================
# ĐÁNH GIÁ TỪNG MÔ HÌNH ĐƠN LẺ (CLASS WEIGHT)
# ==========================================
print(f"{'MÔ HÌNH':<25} | {'ROC-AUC':<10} | {'F1-SCORE (MACRO)'}")
print("-" * 55)

# Đưa các mô hình vào một Dictionary để chạy vòng lặp cho nhàn
models_dict = {
    'Random Forest': rf_model,
    'SVM': svm_model,
    'Logistic Regression': lr_model,
    'XGBoost': xgb_model,
    'Decision Tree': dt_model
}

for name, model in models_dict.items():
    # 1. Lấy điểm rủi ro (AUC)
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    
    # 2. Lấy nhãn dự đoán cứng (F1-score Macro)
    # Lưu ý: Dùng .predict() để lấy nhãn 0/1 do mô hình tự quyết định
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro') # Dùng Macro F1 cho công bằng
    
    # In ra dạng bảng
    print(f"{name:<25} | {auc:<10.4f} | {f1:.4f}")

MÔ HÌNH                   | ROC-AUC    | F1-SCORE (MACRO)
-------------------------------------------------------
Random Forest             | 0.8638     | 0.7554


d:\DS102\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
d:\DS102\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
d:\DS102\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
d:\DS102\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


SVM                       | 0.8566     | 0.7348
Logistic Regression       | 0.7700     | 0.6434
XGBoost                   | 0.8622     | 0.7492
Decision Tree             | 0.8486     | 0.7198


In [11]:
import joblib
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import VotingClassifier
from scipy.stats import ttest_rel
import numpy as np

# ==========================================
# 1. LOAD MODEL & LẤY MÔ HÌNH TỐT NHẤT
# ==========================================
rf_saved = joblib.load('../models/rf_weighted.sav')
svm_saved = joblib.load('../models/svm_weighted.sav')
xgb_saved = joblib.load('../models/xgb_weighted.sav')
dt_saved = joblib.load('../models/dt_weighted.sav')
lr_saved = joblib.load('../models/lr_weighted.sav') # Lấy LR làm Baseline

# Trích xuất 'best_estimator_' (cái vỏ xịn nhất) ra khỏi GridSearch
def get_best(model):
    return model.best_estimator_ if hasattr(model, 'best_estimator_') else model

rf_best = get_best(rf_saved)
svm_best = get_best(svm_saved)
xgb_best = get_best(xgb_saved)
dt_best = get_best(dt_saved)
lr_best = get_best(lr_saved)

# ==========================================
# 2. KHỞI TẠO VOTING CLASSIFIER CHUẨN SKLEARN
# ==========================================
# Đóng gói 4 mô hình của bạn vào 1 cái hộp Ensemble để Scikit-learn có thể chạy tự động
ensemble_model = VotingClassifier(
    estimators=[
        ('rf', rf_best),
        ('svm', svm_best),
        ('xgb', xgb_best),
        ('dt', dt_best)
    ],
    voting='soft'
)

# ==========================================
# 3. CHẠY 10-FOLD CROSS VALIDATION
# ==========================================
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# LƯU Ý: Chỗ này truyền biến X và y TỔNG (Dữ liệu chưa cắt train/test)
# Thêm n_jobs=-1 để ép máy tính dùng toàn bộ CPU chạy cho nhanh
print("⏳ Đang cày 10-fold cho Baseline (Logistic Regression)...")
scores_lr = cross_val_score(lr_best, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print("⏳ Đang cày 10-fold cho SOTA (XGBoost)...")
scores_xgb = cross_val_score(xgb_best, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print("⏳ Đang cày 10-fold cho Ensemble (Sẽ hơi lâu, kiên nhẫn chút nhé)...")
scores_ensemble = cross_val_score(ensemble_model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

# ==========================================
# 4. CHẠY PAIRED T-TEST & IN BÁO CÁO
# ==========================================
print("\n" + "="*60)
print("🏆 KẾT QUẢ KIỂM ĐỊNH THỐNG KÊ (PAIRED T-TEST)")
print("="*60)

# TRẬN 1: Đánh bại sách giáo khoa
t_stat_1, p_val_1 = ttest_rel(scores_ensemble, scores_lr)
print("\n[TRẬN 1] Mô hình đề xuất (Ensemble) vs Mô hình cơ sở (Logistic Regression)")
print(f" -> Điểm trung bình: Ensemble ({scores_ensemble.mean():.4f}) | LR ({scores_lr.mean():.4f})")
print(f" -> p-value: {p_val_1:.5f}")
if p_val_1 < 0.05:
    print(" => KẾT LUẬN: Bác bỏ giả thuyết H0. Ensemble THẮNG ÁP ĐẢO (Có ý nghĩa thống kê).")
else:
    print(" => KẾT LUẬN: Không đủ cơ sở để nói Ensemble tốt hơn (Chênh lệch do ngẫu nhiên).")

# ==========================================
# CHẠY 10-FOLD CHO SVM (Chèn vào phần 3)
# ==========================================
print("⏳ Đang cày 10-fold cho Non-linear (SVM)... (Có thể hơi lâu nhé)")
scores_svm = cross_val_score(svm_best, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

# ==========================================
# TRẬN 1.5 (Chèn vào phần 4)
# ==========================================
t_stat_svm, p_val_svm = ttest_rel(scores_ensemble, scores_svm)
print("\n[TRẬN GIỮA] Mô hình đề xuất (Ensemble) vs Mô hình Phi tuyến (SVM)")
print(f" -> Điểm trung bình: Ensemble ({scores_ensemble.mean():.4f}) | SVM ({scores_svm.mean():.4f})")
print(f" -> p-value: {p_val_svm:.5f}")

if p_val_svm < 0.05 and scores_ensemble.mean() > scores_svm.mean():
    print(" => KẾT LUẬN: Bác bỏ H0. Ensemble THẮNG SVM (Có ý nghĩa thống kê).")
elif p_val_svm < 0.05 and scores_ensemble.mean() < scores_svm.mean():
    print(" => KẾT LUẬN: Bác bỏ H0. Ensemble THUA SVM.")
else:
    print(" => KẾT LUẬN: HÒA. Không có sự khác biệt ý nghĩa về mặt thống kê.")    

# TRẬN 2: Chung kết thế giới SOTA
t_stat_2, p_val_2 = ttest_rel(scores_ensemble, scores_xgb)
print("\n[TRẬN 2] Mô hình đề xuất (Ensemble) vs Mô hình mạnh nhất (XGBoost)")
print(f" -> Điểm trung bình: Ensemble ({scores_ensemble.mean():.4f}) | XGBoost ({scores_xgb.mean():.4f})")
print(f" -> p-value: {p_val_2:.5f}")
if p_val_2 < 0.05 and scores_ensemble.mean() > scores_xgb.mean():
    print(" => KẾT LUẬN: Bác bỏ giả thuyết H0. Ensemble ĐÃ ĐÁNH BẠI XGBoost (Có ý nghĩa thống kê).")
elif p_val_2 < 0.05 and scores_ensemble.mean() < scores_xgb.mean():
    print(" => KẾT LUẬN: Bác bỏ giả thuyết H0. Ensemble THUA XGBoost.")
else:
    print(" => KẾT LUẬN: HÒA. Sự chênh lệch giữa Ensemble và XGBoost không có ý nghĩa thống kê (Hiệu suất tương đương).")

⏳ Đang cày 10-fold cho Baseline (Logistic Regression)...
⏳ Đang cày 10-fold cho SOTA (XGBoost)...
⏳ Đang cày 10-fold cho Ensemble (Sẽ hơi lâu, kiên nhẫn chút nhé)...

🏆 KẾT QUẢ KIỂM ĐỊNH THỐNG KÊ (PAIRED T-TEST)

[TRẬN 1] Mô hình đề xuất (Ensemble) vs Mô hình cơ sở (Logistic Regression)
 -> Điểm trung bình: Ensemble (0.8648) | LR (0.7690)
 -> p-value: 0.00000
 => KẾT LUẬN: Bác bỏ giả thuyết H0. Ensemble THẮNG ÁP ĐẢO (Có ý nghĩa thống kê).
⏳ Đang cày 10-fold cho Non-linear (SVM)... (Có thể hơi lâu nhé)

[TRẬN GIỮA] Mô hình đề xuất (Ensemble) vs Mô hình Phi tuyến (SVM)
 -> Điểm trung bình: Ensemble (0.8648) | SVM (0.8493)
 -> p-value: 0.00010
 => KẾT LUẬN: Bác bỏ H0. Ensemble THẮNG SVM (Có ý nghĩa thống kê).

[TRẬN 2] Mô hình đề xuất (Ensemble) vs Mô hình mạnh nhất (XGBoost)
 -> Điểm trung bình: Ensemble (0.8648) | XGBoost (0.8617)
 -> p-value: 0.06655
 => KẾT LUẬN: HÒA. Sự chênh lệch giữa Ensemble và XGBoost không có ý nghĩa thống kê (Hiệu suất tương đương).
